In [6]:
%pip install -q nltk


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [31]:
import os
os.environ['NLTK_ALLOW_PROXIED_URLOPEN'] = '1'

import re
import math
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
russian_stopwords = stopwords.words('russian')
from collections import Counter
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /Users/alena/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/alena/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [32]:
corpus = open('data/lenta.txt', encoding='utf-8').read()

### Задание 1

In [33]:
def ngrammer(tokens, n=2):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngrams.append(' '.join(tokens[i:i+n]))
    return ngrams

In [34]:
tokens = word_tokenize(
    corpus,
    language='russian',
    preserve_line=True
)

tokens = [
    token.lower()
    for token in tokens
    if not re.match(r'\W+', token)
]

fivegram_counts_1 = Counter(
    ngrammer(tokens, 5)
)

top100_1 = fivegram_counts_1.most_common(100)

print('Первые 10 5-грамм без учета границ предложений:')
for ngram, count in top100_1[:10]:
    print(count, ngram)

Первые 10 5-грамм без учета границ предложений:
400 риа новости со ссылкой на
320 сообщает риа новости со ссылкой
196 как сообщили риа новости в
149 как сообщает риа новости со
142 сообщает интерфакс со ссылкой на
118 сообщает итар-тасс со ссылкой на
113 об этом риа новости сообщили
104 об этом сообщает риа новости
99 этом риа новости сообщили в
93 со ссылкой на источники в


In [35]:
sentences = sent_tokenize(
    corpus,
    language='russian'
)

tokenized_sentences = [
    word_tokenize(sentence)
    for sentence in sentences
]

tokenized_sentences = [
    [
        token.lower()
        for token in sentence
        if not re.match(r'\W+', token)
    ]
    for sentence in tokenized_sentences
]

fivegram_counts_2 = Counter()

for sentence in tokenized_sentences:
    fivegram_counts_2.update(
        ngrammer(sentence, 5)
    )

top100_2 = fivegram_counts_2.most_common(100)

print('Первые 10 5-грамм с учетом границ предложений:')
for ngram, count in top100_2[:10]:
    print(count, ngram)

Первые 10 5-грамм с учетом границ предложений:
400 риа новости со ссылкой на
320 сообщает риа новости со ссылкой
196 как сообщили риа новости в
149 как сообщает риа новости со
142 сообщает интерфакс со ссылкой на
118 сообщает итар-тасс со ссылкой на
113 об этом риа новости сообщили
104 об этом сообщает риа новости
99 этом риа новости сообщили в
93 со ссылкой на источники в


In [36]:
set1 = {ngram for ngram, count in top100_1}
set2 = {ngram for ngram, count in top100_2}

print('\nТолько в первом топ-100:')
print(set1 - set2)

print('\nТолько во втором топ-100:')
print(set2 - set1)


Только в первом топ-100:
{'сообщает риа новости по данным', 'группировки войск на северном кавказе.', 'сообщает риа новости по словам', 'по борьбе с экономическими преступлениями', 'передает риа новости по словам', 'федеральных сил на северном кавказе.'}

Только во втором топ-100:
{'о защите чести и достоинства', 'объединенной группировки на северном кавказе', 'в пресс-центре объединенной группировки войск', 'с 1 января 2001 года', 'контртеррористической операции на северном кавказе', 'представителя правительства рф в чечне'}


### Задание 2

![Описание картинки](IMG1.png)

### Задание 3

In [54]:
def ngrammer(tokens, n=2, stops=set()):
    ngrams = []
    tokens = [token for token in tokens if token not in stops]
    for i in range(0, len(tokens)-n+1):
        ngrams.append(' '.join(tokens[i:i+n]))
    return ngrams

def collect_stats(corpus, stops):
    unigrams = Counter()
    bigrams = Counter()
    for sent in corpus:
        unigrams.update(sent)
        bigrams.update(ngrammer(sent, 2, stops))
    return unigrams, bigrams

unigrams, bigrams = collect_stats(tokenized_sentences, russian_stopwords)

In [55]:
# Dice
def scorer(word_count_a, word_count_b, bigram_count, min_count=0):
    try:
        score = (2*(bigram_count - min_count) / ((word_count_a + word_count_b)))
    except ZeroDivisionError:
        return 0
    return score

def score_bigrams(unigrams, bigrams, scorer, threshold=-100000, min_count=0, **kwargs):
    bigram2score = Counter()
    for bigram in bigrams:
        word_a, word_b = bigram.split()
        score = scorer(unigrams[word_a], unigrams[word_b], bigrams[bigram], min_count=min_count, **kwargs)
        if score > threshold:
            bigram2score[bigram] = score
    return bigram2score

bigram2score_dice = score_bigrams(unigrams, bigrams, scorer, min_count=20)

In [56]:
# PMI
def scorer_pmi(word_count_a, word_count_b, bigram_count, N, min_count=0):
    try:
        score = math.log2(
            ((bigram_count - min_count) * N) /
            (word_count_a * word_count_b)
        )
    except (ZeroDivisionError, ValueError):
        return 0
    return score
    
def score_bigrams_pmi(unigrams, bigrams, scorer, threshold=-100000, min_count=0):
    bigram2score = Counter()
    N = sum(unigrams.values())
    for bigram in bigrams:
        word_a, word_b = bigram.split()
        score = scorer(unigrams[word_a], unigrams[word_b], bigrams[bigram], N, min_count)
        if score > threshold:
            bigram2score[bigram] = score
    return bigram2score

bigram2score_pmi = score_bigrams_pmi(unigrams, bigrams, scorer_pmi, min_count=20)

In [57]:
dice_top30 = bigram2score_dice.most_common(30)
pmi_top30 = bigram2score_pmi.most_common(30)

print("=== TOP-30 Dice ===")
for bg, sc in dice_top30:
    print(f"{bg:30} {sc:.4f}")

print("\n=== TOP-30 PMI ===")
for bg, sc in pmi_top30:
    print(f"{bg:30} {sc:.4f}")

set_dice = {bg for bg, _ in dice_top30}
set_pmi  = {bg for bg, _ in pmi_top30}

print("Общие биграммы:", sorted(set_dice & set_pmi))

=== TOP-30 Dice ===
риа новости                    0.9801
северном кавказе               0.8911
associated press               0.8692
new york                       0.8436
сих пор                        0.7818
взрывное устройство            0.7332
таким образом                  0.7314
рао еэс                        0.6791
доменных имен                  0.6303
чрезвычайным ситуациям         0.6187
налогам сборам                 0.6040
wall street                    0.6038
населенного пункта             0.6027
объединенной группировки       0.5987
возбуждено уголовное           0.5967
мадлен олбрайт                 0.5949
прошлой неделе                 0.5801
владимиром путиным             0.5798
внутренних дел                 0.5696
точки зрения                   0.5657
великой отечественной          0.5556
правоохранительных органов     0.5498
уголовное дело                 0.5453
washington post                0.5430
правоохранительные органы      0.5393
bank of                       

### Задание 4

In [74]:
def merge_round(corpus, min_count=5, n_merges=1000, scorer=scorer_pmi):
    u = Counter(t for s in corpus for t in s)
    b = Counter()
    for s in corpus:
        b.update(zip(s, s[1:]))

    N = sum(u.values())

    scores = {}
    for (w1, w2), c in b.items():
        if c >= min_count:
            scores[(w1, w2)] = scorer(u[w1], u[w2], c, N, min_count)

    best = set(sorted(scores, key=scores.get, reverse=True)[:n_merges])

    merged = []
    for s in corpus:
        ws = []
        i = 0
        while i < len(s):
            if i < len(s) - 1 and (s[i], s[i + 1]) in best:
                ws.append('_'.join((s[i], s[i + 1])))
                i += 2
            else:
                ws.append(s[i])
                i += 1
        merged.append(ws)

    return merged, sorted(scores, key=scores.get, reverse=True)[:n_merges]

In [75]:
_, merges = merge_round(sub_corpus, min_count=1, n_merges=1000)

In [76]:
sub_corpus = [list(s) for s in tokenized_sentences[:30000]]
base_len = sum(len(s) for s in sub_corpus)

saved_merges = []
for round_i in range(1, 10):
    sub_corpus, merges = merge_round(sub_corpus, min_count=5, n_merges=1000)
    if not merges:
        break
    saved_merges.append(merges)
    cur_len = sum(len(s) for s in sub_corpus)
    vocab_now = set(t for s in sub_corpus for t in s)
    print(f'раунд {round_i}: {cur_len:>8,} токенов ({cur_len/base_len:5.1%}), '
          f'топ-слияние: {merges[0][0]}_{merges[0][1]}')

раунд 1:  593,132 токенов (97.5%), топ-слияние: philip_morris
раунд 2:  574,028 токенов (94.3%), топ-слияние: corriere_della_sera
раунд 3:  557,379 токенов (91.6%), топ-слияние: получили_ранения_различной_степени_тяжести
раунд 4:  538,745 токенов (88.5%), топ-слияние: подписку_о_невыезде
раунд 5:  522,942 токенов (85.9%), топ-слияние: оставляет_за_собой_право
раунд 6:  502,913 токенов (82.7%), топ-слияние: по_делам_печати_телерадиовещания_и_средств_массовых_коммуникаций
раунд 7:  491,327 токенов (80.8%), топ-слияние: го_и_чс
раунд 8:  482,778 токенов (79.3%), топ-слияние: в_связи_с_переходом_на_другую_работу
раунд 9:  477,866 токенов (78.5%), топ-слияние: авиация_и_артиллерия


In [77]:
def apply_merges(tokens, saved_merges):
    cur = list(tokens)
    for round_merges in saved_merges:
        merge_set = set(round_merges)
        merged = []
        i = 0
        while i < len(cur):
            if i < len(cur) - 1 and (cur[i], cur[i + 1]) in merge_set:
                merged.append('_'.join((cur[i], cur[i + 1])))
                i += 2
            else:
                merged.append(cur[i])
                i += 1
        cur = merged
    return cur

def normalize(text):
    return [tok.lower() for tok in word_tokenize(text) if not re.match(r'\W+', tok)]

In [79]:
test_sentences = [
    'Об этом сообщает РИА Новости со ссылкой на источник в администрации президента.',
    'Боевые действия продолжаются на северном Кавказе уже вторую неделю.',
    'Wall Street Journal опубликовал статью о состоянии рынка нефти.',
    'Сотрудника уволили в связи с переходом на другую работу.',
    'Совет директоров компании собрался на внеочередное заседание.',
    'Wall Street Journal опубликовала статью',
    'Как сообщает РИА Новости со ссылкой на источник',
    'События произошли в Восточном Тиморе',
    'Это произошло на прошлой неделе'
]

for sent in test_sentences:
    toks = normalize(sent)
    merged = apply_merges(toks, saved_merges)
    print('ИСХОДНО: ', ' '.join(toks))
    print('СМЕРЖЕНО:', ' '.join(merged))

ИСХОДНО:  об этом сообщает риа новости со ссылкой на источник в администрации президента
СМЕРЖЕНО: об_этом_сообщает_риа_новости_со_ссылкой_на источник_в администрации_президента
ИСХОДНО:  боевые действия продолжаются на северном кавказе уже вторую неделю
СМЕРЖЕНО: боевые_действия продолжаются на_северном_кавказе уже вторую неделю
ИСХОДНО:  wall street journal опубликовал статью о состоянии рынка нефти
СМЕРЖЕНО: wall_street_journal опубликовал статью о_состоянии рынка нефти
ИСХОДНО:  сотрудника уволили в связи с переходом на другую работу
СМЕРЖЕНО: сотрудника уволили в_связи_с_переходом_на_другую_работу
ИСХОДНО:  совет директоров компании собрался на внеочередное заседание
СМЕРЖЕНО: совет_директоров компании собрался на внеочередное заседание
ИСХОДНО:  wall street journal опубликовала статью
СМЕРЖЕНО: wall_street_journal опубликовала статью
ИСХОДНО:  как сообщает риа новости со ссылкой на источник
СМЕРЖЕНО: как_сообщает_риа_новости_со_ссылкой_на источник
ИСХОДНО:  события произошли в во

### Задание 5

Когда сервис называют полностью автоматизированным, естественно предположить, что участие человека в его работе сведено к минимуму. Однако в случае автономного транспорта не совсем понятно, где именно должна проходить граница между автоматизацией отдельной функции и автоматизацией всего сервиса. Машина может самостоятельно перевозить пассажиров, но при этом зависеть от людей, которые готовят её к работе, следят за её состоянием и решают возникающие проблемы. Именно это противоречие показалось мне самым интересным в статье Bay Area Current о работе сотрудников, обслуживающих автомобили Waymo. Компания рекламирует свои автомобили лозунгом "Welcome to the future" и преподносит поездку как полностью автоматизированную. Для пассажира это действительно так: он заказывает машину, садится в неё и едет без водителя, не взаимодействуя с человеком. Однако статья показывает, что отсутствие человека в салоне совсем не означает его отсутствия в  системе. Именно это заставило меня задуматься о том, что вообще следует считать автономностью.

С одной стороны, мне кажется неправильным использовать существование обслуживающего персонала как доказательство того, что технология Waymo не является автономной. Автомобиль действительно самостоятельно выполняет основную задачу водителя - перемещается по дорогам и перевозит пассажиров. То, что его приходится заряжать, мыть, ремонтировать или чистить сенсоры, само по себе этого не отменяет. С другой стороны, статья показывает гораздо более широкую инфраструктуру человеческого труда, чем можно представить, просто увидев беспилотное такси на улице. Из интервью становится понятно, что сотрудники уровня L1 занимаются подготовкой автомобилей к работе, зарядкой, проверкой состояния машин, очисткой сенсоров и устранением простых неисправностей. Иногда их техническая работа действительно сводится к тому, чтобы выключить автомобиль, подождать и снова включить его. Другие сотрудники занимаются уборкой салонов, если пассажиры оставляют после себя мусор, рвоту или другие загрязнения. Кроме того, есть выездные работники, которые приезжают к автомобилю, если нужно физически закрыть дверь, переместить машину или организовать её эвакуацию. В отдельных случаях используются и удалённые сотрудники, которые могут подключаться к автомобилю и помогать при возникновении проблем. Всё это заставляет задаться вопросом, насколько автономным является не отдельный автомобиль, а сам сервис.

Автономным можно назвать сам процесс вождения, даже если машину обслуживают люди. Но если слово "автономный" воспринимается как характеристика всего сервиса, ситуация становится менее однозначной: за простым для пользователя интерфейсом существует довольно сложная система из работников, подрядчиков, менеджеров и служб поддержки. Особенно точно это описывает фраза сотрудника о том, что людей нанимают скорее за их способность выполнить работу в нужный момент, чем за сам объём постоянно выполняемых задач. В такой модели роль человека меняется: он уже не обязательно постоянно участвует в работе системы, а нужен прежде всего для решения ситуаций, с которыми автоматика не справляется сама. Чем надёжнее становится технология, тем реже может требоваться его вмешательство, но сама необходимость такого человеческого резерва пока не исчезает.

При этом сама статья, на мой взгляд, даёт довольно одностороннюю картину происходящего. Главный источник информации - один анонимный сотрудник подрядчика Transdev, который, по собственным словам, практически не взаимодействует с сотрудниками Waymo и не видит работу компании целиком. Его рассказ хорошо показывает устройство работы на уровне депо, но гораздо хуже подходит для выводов о сервисе в целом. В статье почти нет количественных данных: неизвестно, сколько сотрудников приходится на определённое количество автомобилей, как часто возникают ситуации, требующие ручного вмешательства, и меняется ли эта частота по мере развития технологии. Поэтому трудно определить, является ли описанная человеческая инфраструктура постоянной опорой сервиса или в основном страховкой на случай сравнительно редких сбоев. Без таких данных довольно трудно делать вывод о реальной степени автономности Waymo.

Отдельного внимания заслуживает то, что отсутствие водителя может менять поведение самих пассажиров. В статье постоянно возникают истории о мусоре, курении, рвоте, экскрементах и даже забытых в салоне свиных головах. Если рядом находится водитель, пассажир понимает, что последствия его поведения увидит конкретный человек. В беспилотном автомобиле эта социальная связь исчезает, и салон легче начать воспринимать как безличное пространство. Возможно, именно поэтому некоторые люди ведут себя так, как не позволили бы себе в присутствии другого человека. Однако последствия этого поведения всё равно не исчезают - просто с ними позже приходится иметь дело сотруднику, которого пассажир никогда не увидит.

Думаю, дальнейшее развитие Waymo и подобных сервисов будет зависеть не только от улучшения системы автономного вождения. Waymo действительно убрала водителя из автомобиля, но не человека из системы. Для настоящего сокращения человеческого участия придётся автоматизировать или хотя бы упростить обслуживание автомобилей, диагностику неисправностей и часть процессов внутри депо. Полностью избавиться от людей вряд ли получится, потому что физический мир слишком непредсказуем, а некоторые задачи вроде уборки повреждённого или загрязнённого салона плохо поддаются автоматизации. Но один сотрудник потенциально сможет обслуживать всё больше автомобилей, если сами машины будут реже требовать вмешательства. Поэтому в будущем подобным компаниям придётся пересмотреть роль человека в системе и оптимизировать его работу.